In [4]:
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2Model, GPT2LMHeadModel
from torch.nn.functional import softmax
from datasets import load_dataset


/Users/plato/code/interpret/v/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

In [6]:
LABELS = ['negative', 'positive']
TOKENIZED_LABELS = list(tokenizer(LABELS, return_tensors='pt', padding=True, truncation=True)['input_ids'].squeeze().tolist())
LABEL_MAPPING = {'negative': 0, 'positive': 1}


In [7]:
imdb = load_dataset('imdb')
imdb

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [8]:
imdb_test = imdb['test'].shuffle().select(range(1000))
# imdb_test['label']

In [9]:
def classify_sentiment(model, batch):
    with torch.no_grad():
        outputs = model(input_ids=batch)
    logits = outputs.logits[:, -1, :]
    probabilities = softmax(logits, dim=-1)
    probabilities = probabilities[:, TOKENIZED_LABELS]
    predicted_idxs = torch.argmax(probabilities, dim=-1)
    predicted_sentiments = [LABELS[idx.item()] for idx in predicted_idxs]
    return predicted_sentiments

def classify_sentiment_generate(model, batch):
    outputs = model.generate(input_ids=batch, max_length=batch.shape[1] + 1, num_return_sequences=1)
    decoded_outputs = [tokenizer.decode(output) for output in outputs]
    predicted_sentiments = []
    for decoded_output in decoded_outputs:
        last_word = decoded_output.split()[-1].lower()
        if last_word in LABELS:
            predicted_sentiments.append(last_word)
        else:
            predicted_sentiments.append(classify_sentiment_random(model, [decoded_output])[0])
    return predicted_sentiments

def classify_sentiment_random(model, batch):
    return ['positive' if torch.rand(1) < 0.5 else 'negative' for _ in batch]

In [15]:
from torch.utils.data import DataLoader

def test_classification_methods(model, dataset, methods):
    results = {method_name: {'correct': 0, 'total': 0} for method_name in methods.keys()}

    for batch_samples in dataset:
        batch_texts = [sample + '\n\n the sentiment of this review is ' for sample in batch_samples['text']]
        batch_labels = ['positive' if sample == 1 else 'negative' for sample in batch_samples['label']]

        batch_tokens = tokenizer(batch_texts, return_tensors='pt', max_length=1024, truncation=True, padding=True)['input_ids']

        for method_name, method in methods.items():
            predicted_labels = method(model, batch_tokens)
            for predicted_label, true_label in zip(predicted_labels, batch_labels):
                results[method_name]['correct'] += 1 if predicted_label == true_label else 0
                results[method_name]['total'] += 1

    for method_name in results.keys():
        accuracy = results[method_name]['correct'] / results[method_name]['total']
        print(f"{method_name} accuracy: {accuracy:.2%}")

    return results


imdb_test_dl = DataLoader(imdb_test, batch_size=16, shuffle=False)
methods = {
    'logits': classify_sentiment,
    'random': classify_sentiment_random
}

test_classification_methods(model, imdb_test_dl, methods)


logits accuracy: 48.80%
random accuracy: 48.10%
